# ЛР 01.2 — Wrapper + Embedded методы отбора признаков (TODO)

Ориентир по времени: 90 минут.

## Цель
Сравнить wrapper и embedded подходы, затем собрать 2-3 кандидатных набор признаков (feature set) для итогового сравнения моделей.

In [1]:
# Что делаем: Подключаем библиотеки и настраиваем рабочее окружение ноутбука.
# Зачем: Без корректных импортов и констант следующие шаги не будут воспроизводимыми.
# Как читать результат: После выполнения этой ячейки не должно быть ошибок импорта; переменные окружения должны появиться в памяти.
# Типичные ошибки: Частая ошибка — запускать следующие ячейки до инициализации импортов и путей к данным.

# Подключаем зависимости для этого шага.
from pathlib import Path
import sys
import json

import numpy as np
import pandas as pd

from sklearn.ensemble import RandomForestClassifier
from sklearn.feature_selection import RFE, SequentialFeatureSelector
from sklearn.inspection import permutation_importance
from sklearn.linear_model import LogisticRegression

ROOT = Path('..') if (Path('..') / 'lab_utils.py').exists() else Path('.')
ROOT = ROOT.resolve()
if str(ROOT) not in sys.path:
    sys.path.append(str(ROOT))

# Подключаем зависимости для этого шага.
from lab_utils import (
    load_dataset,
    split_xy,
    train_test_split_stratified,
    build_preprocessor,
    transform_with_names,
    to_dense,
    append_ranking_rows,
    build_shortlist,
)

SEED = 42
DATASETS = {
    'medical': ROOT / 'data' / 'medical_cardiovascular_risk.csv',
    'finance': ROOT / 'data' / 'finance_credit_risk.csv',
}
OUTPUT_DIR = ROOT / 'outputs'
OUTPUT_DIR.mkdir(exist_ok=True)

## Подготовка данных и загрузка shortlist из Notebook 1
Если `shortlist_filter.json` уже есть, ограничиваем пространство поиска для ускорения wrapper-процедур.

In [2]:
# Что делаем: Загружаем входные данные и артефакты предыдущих шагов.
# Зачем: Этот шаг задает исходный контекст: без него метрики и графики будут считаться по неверным данным.
# Как читать результат: Проверьте размеры таблиц и названия ключевых колонок: это главный индикатор корректной загрузки.
# Типичные ошибки: Частая ошибка — использовать не тот файл или устаревший артефакт из другой лабораторной работы.

filter_shortlist_path = OUTPUT_DIR / 'shortlist_filter.json'
if filter_shortlist_path.exists():
    with open(filter_shortlist_path, 'r', encoding='utf-8') as f:
        filter_shortlists = json.load(f)
else:
    filter_shortlists = {}

print('shortlist loaded:', bool(filter_shortlists))

shortlist loaded: True


In [3]:
# Что делаем: Загружаем входные данные и артефакты предыдущих шагов.
# Зачем: Этот шаг задает исходный контекст: без него метрики и графики будут считаться по неверным данным.
# Как читать результат: Проверьте размеры таблиц и названия ключевых колонок: это главный индикатор корректной загрузки.
# Типичные ошибки: Частая ошибка — использовать не тот файл или устаревший артефакт из другой лабораторной работы.

prepared = {}
# Итерируемся по объектам и последовательно накапливаем результаты.
for dataset_name, path in DATASETS.items():
    df = load_dataset(str(path))
    X, y = split_xy(df)
    X_train, X_test, y_train, y_test = train_test_split_stratified(X, y, random_state=SEED)

    preprocessor = build_preprocessor(X_train)
    X_train_t, X_test_t, feature_names = transform_with_names(preprocessor, X_train, X_test)

    X_train_t = to_dense(X_train_t)
    X_test_t = to_dense(X_test_t)
    feature_names = np.array(feature_names)

    preferred = filter_shortlists.get(dataset_name, [])
    preferred = [feat for feat in preferred if feat in set(feature_names)]

    if len(preferred) >= 8:
        idx = [int(np.where(feature_names == f)[0][0]) for f in preferred]
        X_train_pool = X_train_t[:, idx]
        X_test_pool = X_test_t[:, idx]
        pool_features = feature_names[idx]
    else:
        X_train_pool = X_train_t
        X_test_pool = X_test_t
        pool_features = feature_names

    prepared[dataset_name] = {
        'X_train': X_train_pool,
        'X_test': X_test_pool,
        'y_train': y_train,
        'y_test': y_test,
        'feature_names': pool_features,
    }

    print(f"{dataset_name}: pool_features={len(pool_features)}")

medical: pool_features=12
finance: pool_features=12


## Методы отбора
- `RFE(LogisticRegression)`
- `SequentialFeatureSelector(LogisticRegression)`
- L1-регуляризация (`LogisticRegression`, `penalty='l1'`)
- `RandomForest` feature importance
- permutation importance

TODO: поэкспериментируйте с `n_features_to_select` и сравните стабильность топа.

In [4]:
# Что делаем: Обучаем модель и, при необходимости, подбираем параметры.
# Зачем: На этом шаге формируется качество модели, которое дальше анализируется в метриках и графиках.
# Как читать результат: Смотрите на итоговые метрики и выбранные параметры: они должны соответствовать ожиданиям шага.
# Типичные ошибки: Частая ошибка — случайно обучить модель на неправильном split и получить смещенную оценку качества.

ranking_rows = []

# Итерируемся по объектам и последовательно накапливаем результаты.
for dataset_name, bundle in prepared.items():
    X_train = bundle['X_train']
    X_test = bundle['X_test']
    y_train = bundle['y_train']
    y_test = bundle['y_test']
    feature_names = bundle['feature_names'].tolist()

    n_features_to_select = min(10, max(4, X_train.shape[1] // 2))

    base_lr = LogisticRegression(
        max_iter=2500,
        class_weight='balanced',
        random_state=SEED,
        solver='liblinear',
    )

    # 1) RFE
    rfe = RFE(estimator=base_lr, n_features_to_select=n_features_to_select)
    # Обучаем модель на подготовленных данных.
    rfe.fit(X_train, y_train)
    rfe_scores = 1.0 / rfe.ranking_
    append_ranking_rows(ranking_rows, dataset_name, 'rfe', feature_names, rfe_scores)

    # 2) SequentialFeatureSelector
    sfs = SequentialFeatureSelector(
        estimator=base_lr,
        n_features_to_select=n_features_to_select,
        direction='forward',
        scoring='f1',
        cv=4,
        n_jobs=-1,
    )
    # Обучаем модель на подготовленных данных.
    sfs.fit(X_train, y_train)
    sfs_scores = sfs.get_support().astype(float)
    append_ranking_rows(ranking_rows, dataset_name, 'sfs_forward', feature_names, sfs_scores)

    # 3) L1 coefficients
    l1_model = LogisticRegression(
        penalty='l1',
        solver='liblinear',
        max_iter=3000,
        class_weight='balanced',
        random_state=SEED,
    )
    # Обучаем модель на подготовленных данных.
    l1_model.fit(X_train, y_train)
    l1_scores = np.abs(l1_model.coef_[0])
    append_ranking_rows(ranking_rows, dataset_name, 'l1_logreg', feature_names, l1_scores)

    # 4) RandomForest feature importance
    rf = RandomForestClassifier(
        n_estimators=300,
        random_state=SEED,
        class_weight='balanced_subsample',
        n_jobs=-1,
    )
    # Обучаем модель на подготовленных данных.
    rf.fit(X_train, y_train)
    rf_scores = rf.feature_importances_
    append_ranking_rows(ranking_rows, dataset_name, 'rf_importance', feature_names, rf_scores)

    # 5) Permutation importance
    perm = permutation_importance(
        estimator=rf,
        X=X_test,
        y=y_test,
        n_repeats=8,
        random_state=SEED,
        scoring='f1',
        n_jobs=-1,
    )
    perm_scores = np.maximum(perm.importances_mean, 0.0)
    append_ranking_rows(ranking_rows, dataset_name, 'permutation', feature_names, perm_scores)

feature_ranking = (
    pd.DataFrame(ranking_rows)
    .sort_values(['dataset', 'method', 'rank'])
    .reset_index(drop=True)
)
feature_ranking.head(20)

,dataset,method,feature,score,rank
0,finance,l1_logreg,num__loan_to_income,0.378214,1
1,finance,l1_logreg,num__credit_score,0.360359,2
2,finance,l1_logreg,num__annual_income,0.280105,3
3,finance,l1_logreg,cat__previous_default_no,0.255868,4
4,finance,l1_logreg,num__delinquency_count,0.230762,5
5,finance,l1_logreg,cat__previous_default_yes,0.209766,6
6,finance,l1_logreg,cat__housing_status_rent,0.199111,7
7,finance,l1_logreg,num__utilization_ratio,0.195604,8
8,finance,l1_logreg,num__savings_balance,0.149644,9
9,finance,l1_logreg,num__loan_amount,0.099236,10


## Формирование 2-3 кандидатных наборов признаков
- `set_A_wrapper` на основе RFE/SFS/L1
- `set_B_tree` на основе RF/permutation
- `set_C_hybrid` как гибрид/пересечение

In [5]:
# Что делаем: Выполняем очередной вычислительный блок текущего шага лабораторной работы.
# Зачем: Этот блок готовит промежуточный результат, который используется в следующей ячейке.
# Как читать результат: После выполнения проверьте вывод и убедитесь, что значения выглядят реалистично.
# Типичные ошибки: Частая ошибка — переходить дальше без проверки промежуточного результата.

feature_sets = {}

# Итерируемся по объектам и последовательно накапливаем результаты.
for dataset_name in DATASETS:
    set_a = build_shortlist(
        feature_ranking,
        dataset_name,
        methods=['rfe', 'sfs_forward', 'l1_logreg'],
        top_n=10,
    )
    set_b = build_shortlist(
        feature_ranking,
        dataset_name,
        methods=['rf_importance', 'permutation'],
        top_n=10,
    )

    inter = [f for f in set_a if f in set_b]
    if len(inter) < 8:
        # Итерируемся по объектам и последовательно накапливаем результаты.
        for feat in set_a + set_b:
            if feat not in inter:
                inter.append(feat)
            if len(inter) >= 10:
                break

    feature_sets[dataset_name] = {
        'set_A_wrapper': set_a,
        'set_B_tree': set_b,
        'set_C_hybrid': inter[:10],
    }

feature_sets

{'medical': {'set_A_wrapper': ['num__age',
   'num__cholesterol',
   'num__systolic_bp',
   'num__physical_activity_hours',
   'cat__smoking_status_never',
   'num__stress_level',
   'num__glucose',
   'num__bmi',
   'cat__sex_male',
   'num__diastolic_bp'],
  'set_B_tree': ['num__cholesterol',
   'num__systolic_bp',
   'num__bmi',
   'num__age',
   'num__resting_heart_rate',
   'num__diastolic_bp',
   'num__physical_activity_hours',
   'cat__smoking_status_never',
   'cat__sex_male',
   'num__glucose'],
  'set_C_hybrid': ['num__age',
   'num__cholesterol',
   'num__systolic_bp',
   'num__physical_activity_hours',
   'cat__smoking_status_never',
   'num__glucose',
   'num__bmi',
   'cat__sex_male',
   'num__diastolic_bp']},
 'finance': {'set_A_wrapper': ['num__annual_income',
   'num__credit_score',
   'num__loan_to_income',
   'num__delinquency_count',
   'cat__previous_default_no',
   'num__utilization_ratio',
   'cat__previous_default_yes',
   'cat__housing_status_rent',
   'num__lo

## Самостоятельное изучение по ходу работы

### Что изучено в этом шаге
- Wrapper-методы (RFE, SFS) используют модель как «чёрный ящик» и ищут подмножество признаков, максимизирующее качество модели.
- Embedded-методы (L1-логистическая регрессия, важности Random Forest) встраивают отбор в процесс обучения — быстрее wrapper, но зависят от выбранной модели.
- Permutation importance измеряет падение метрики при перемешивании признака на hold-out — это model-agnostic проверка важности.

### Сравнение с альтернативами
- **RFE vs L1**: RFE точнее для малого числа признаков, но медленнее; L1 даёт разреженное решение за один проход, но чувствителен к масштабу и корреляциям.
- **RF importance vs permutation**: RF importance быстрее, но может завышать важность высококардинальных признаков; permutation дороже, но ближе к реальному вкладу в метрику.

### Источники
- [scikit-learn: RFE](https://scikit-learn.org/stable/modules/generated/sklearn.feature_selection.RFE.html)
- [Permutation importance](https://scikit-learn.org/stable/modules/permutation_importance.html)

### Глоссарий незнакомых терминов (обязательно)
- `Глоссарий обновлен: SequentialFeatureSelector, L1 regularization, Jaccard similarity`


In [6]:
# Что делаем: Загружаем входные данные и артефакты предыдущих шагов.
# Зачем: Этот шаг задает исходный контекст: без него метрики и графики будут считаться по неверным данным.
# Как читать результат: Проверьте размеры таблиц и названия ключевых колонок: это главный индикатор корректной загрузки.
# Типичные ошибки: Частая ошибка — использовать не тот файл или устаревший артефакт из другой лабораторной работы.

feature_ranking_path = OUTPUT_DIR / 'feature_ranking_wrapper_embedded.csv'
feature_sets_path = OUTPUT_DIR / 'feature_sets_wrapper_embedded.json'

# Сохраняем таблицу артефакта в CSV.
feature_ranking.to_csv(feature_ranking_path, index=False)
with open(feature_sets_path, 'w', encoding='utf-8') as f:
    json.dump(feature_sets, f, ensure_ascii=False, indent=2)

print(f'Saved: {feature_ranking_path}')
print(f'Saved: {feature_sets_path}')

Saved: C:\Users\perev\Documents\GitHub\edu-big-data-machine-models\01-feature-importance-and-selection\outputs\feature_ranking_wrapper_embedded.csv
Saved: C:\Users\perev\Documents\GitHub\edu-big-data-machine-models\01-feature-importance-and-selection\outputs\feature_sets_wrapper_embedded.json


## Контрольные точки
1. В `feature_ranking` есть строки для 5 методов.
2. Для каждого датасета сформированы `set_A_wrapper`, `set_B_tree`, `set_C_hybrid`.
3. В каждом наборе минимум 5 признаков.

In [7]:
# Что делаем: Проверяем инварианты и защищаемся от некорректного состояния данных.
# Зачем: Ранняя проверка позволяет поймать ошибку до обучения модели и построения графиков.
# Как читать результат: Если assert срабатывает, сначала проверьте входные данные и только потом продолжайте.
# Типичные ошибки: Частая ошибка — игнорировать сообщения assert и анализировать уже некорректный результат.

required_columns = {'dataset', 'method', 'feature', 'score', 'rank'}
# Проверяем обязательное условие корректности шага.
assert required_columns.issubset(feature_ranking.columns)

# Итерируемся по объектам и последовательно накапливаем результаты.
for dataset_name in DATASETS:
    subset = feature_ranking[feature_ranking['dataset'] == dataset_name]
    # Проверяем обязательное условие корректности шага.
    assert subset['method'].nunique() == 5

    sets = feature_sets[dataset_name]
    # Итерируемся по объектам и последовательно накапливаем результаты.
    for set_name in ['set_A_wrapper', 'set_B_tree', 'set_C_hybrid']:
        # Проверяем обязательное условие корректности шага.
        assert len(sets[set_name]) >= 5

print('Проверки пройдены успешно.')

Проверки пройдены успешно.


## Типичные ошибки
- Слишком большой `n_features_to_select` при малом числе признаков в пуле.
- Неправильная интерпретация L1: нулевые коэффициенты не должны попадать в топ.
- Сравнение permutation importance без фиксированного `random_state`.

## Финальные выводы (заполните)
1. **Пересечение wrapper и embedded**: признаки с высоким средним рангом в RFE/L1 и RF/permutation часто совпадают — это «ядро» устойчиво важных переменных для каждого датасета.
2. **Различия топов**: SFS и permutation могут расходиться из-за стохастики CV и шума на test; L1 обнуляет коррелированные признаки, тогда как tree-based методы распределяют важность между ними.
3. **Кандидат для Notebook 3**: `set_C_hybrid` — компромисс между wrapper- и tree-based наборами; при наличии `set_D_robust` (из самостоятельного задания) его стоит сравнить с hybrid на следующем шаге.

## Обязательные самостоятельные задания (без образца в solutions)

Ниже задания, которые отсутствуют в `solutions` и обязательны к сдаче.
До заполнения этих блоков ноутбук должен останавливаться с `NotImplementedError`.

### Задание 1. Pairwise agreement matrix для методов

**Контракт задания**

Входные переменные:
- `feature_ranking`, список методов `['rfe', 'sfs_forward', 'l1_logreg', 'rf_importance', 'permutation']`.

Ожидаемый выход:
- `method_agreement_long` с колонками:
  `dataset`, `method_a`, `method_b`, `top_k`, `overlap_count`, `jaccard`.

In [ ]:
# Что делаем: Проверяем инварианты и защищаемся от некорректного состояния данных.
# Зачем: Ранняя проверка позволяет поймать ошибку до обучения модели и построения графиков.
# Как читать результат: Если assert срабатывает, сначала проверьте входные данные и только потом продолжайте.
# Типичные ошибки: Частая ошибка — игнорировать сообщения assert и анализировать уже некорректный результат.
# 
required_columns_task1 = [
    'dataset',
    'method_a',
    'method_b',
    'top_k',
    'overlap_count',
    'jaccard',
]
method_agreement_long = pd.DataFrame(columns=required_columns_task1)
assert set(required_columns_task1).issubset(method_agreement_long.columns)

top_k = 10
methods_for_agreement = ['rfe', 'sfs_forward', 'l1_logreg', 'rf_importance', 'permutation']

# TODO(обязательно):
# 1) Для каждого dataset получите top_k признаков для каждого метода.
# 2) Для каждой пары методов посчитайте overlap_count и jaccard.
# 3) Заполните method_agreement_long.


# Обязательный подпункт (методическая рефлексия):
# - До снятия intentional-stop добавьте в релевантный narrative-блок
#   или в `study-notes/*.md` короткое описание: что вы изучили,
#   с чем сравнили, на какие источники опирались.
# - Обновите `study-notes/glossary.md`: добавьте 2-3 термина, встретившихся в этом задании,
#   и укажите их простые объяснения + источники.

agreement_rows = []

for dataset_name in DATASETS:
    ds_ranking = feature_ranking[feature_ranking['dataset'] == dataset_name]
    top_by_method = {}
    for method_name in methods_for_agreement:
        method_subset = (
            ds_ranking[ds_ranking['method'] == method_name]
            .sort_values('rank')
            .head(top_k)
        )
        top_by_method[method_name] = set(method_subset['feature'].tolist())

    for i, method_a in enumerate(methods_for_agreement):
        for method_b in methods_for_agreement[i + 1:]:
            set_a = top_by_method[method_a]
            set_b = top_by_method[method_b]
            inter = set_a & set_b
            union = set_a | set_b
            agreement_rows.append(
                {
                    'dataset': dataset_name,
                    'method_a': method_a,
                    'method_b': method_b,
                    'top_k': top_k,
                    'overlap_count': len(inter),
                    'jaccard': len(inter) / len(union) if union else 0.0,
                }
            )

method_agreement_long = pd.DataFrame(agreement_rows)[required_columns_task1]
print(f'method_agreement_long shape: {method_agreement_long.shape}')
method_agreement_long.head()

method_agreement_long shape: (20, 6)


,dataset,method_a,method_b,top_k,overlap_count,jaccard
0,medical,rfe,sfs_forward,10,8,0.666667
1,medical,rfe,l1_logreg,10,10,1.000000
2,medical,rfe,rf_importance,10,8,0.666667
3,medical,rfe,permutation,10,8,0.666667
4,medical,sfs_forward,l1_logreg,10,8,0.666667


### Задание 2. Стабильность отбора по random_state/ресемплингу

**Контракт задания**

Входные переменные:
- `prepared`, методы отбора (минимум один wrapper и один embedded).

Ожидаемый выход:
- `selection_stability` с колонками:
  `dataset`, `method`, `feature`, `selected_count`, `total_runs`, `stability_rate`.

In [ ]:
# Что делаем: Проверяем инварианты и защищаемся от некорректного состояния данных.
# Зачем: Ранняя проверка позволяет поймать ошибку до обучения модели и построения графиков.
# Как читать результат: Если assert срабатывает, сначала проверьте входные данные и только потом продолжайте.
# Типичные ошибки: Частая ошибка — игнорировать сообщения assert и анализировать уже некорректный результат.
# 
required_columns_task2 = [
    'dataset',
    'method',
    'feature',
    'selected_count',
    'total_runs',
    'stability_rate',
]
selection_stability = pd.DataFrame(columns=required_columns_task2)
assert set(required_columns_task2).issubset(selection_stability.columns)

random_states = [11, 42, 101]

# TODO(обязательно):
# 1) Повторите отбор признаков для нескольких random_state.
# 2) Для каждого feature посчитайте selected_count и stability_rate.
# 3) Заполните selection_stability.


# Обязательный подпункт (методическая рефлексия):
# - До снятия intentional-stop добавьте в релевантный narrative-блок
#   или в `study-notes/*.md` короткое описание: что вы изучили,
#   с чем сравнили, на какие источники опирались.
# - Обновите `study-notes/glossary.md`: добавьте 2-3 термина, встретившихся в этом задании,
#   и укажите их простые объяснения + источники.

stability_methods = ['rfe', 'l1_logreg']
stability_rows = []
total_runs = len(random_states)

for dataset_name, bundle in prepared.items():
    X_train = bundle['X_train']
    y_train = bundle['y_train']
    feature_names = bundle['feature_names'].tolist()
    n_select = min(10, max(4, X_train.shape[1] // 2))

    for method_name in stability_methods:
        selected_counter = {feat: 0 for feat in feature_names}

        for rs in random_states:
            lr = LogisticRegression(
                max_iter=2500,
                class_weight='balanced',
                random_state=rs,
                solver='liblinear',
            )
            if method_name == 'rfe':
                selector = RFE(estimator=lr, n_features_to_select=n_select)
                selector.fit(X_train, y_train)
                selected_feats = [f for f, keep in zip(feature_names, selector.support_) if keep]
            else:
                l1_model = LogisticRegression(
                    penalty='l1',
                    solver='liblinear',
                    max_iter=3000,
                    class_weight='balanced',
                    random_state=rs,
                )
                l1_model.fit(X_train, y_train)
                coef_abs = np.abs(l1_model.coef_[0])
                top_idx = np.argsort(-coef_abs)[:n_select]
                selected_feats = [feature_names[i] for i in top_idx if coef_abs[i] > 1e-8]

            for feat in selected_feats:
                selected_counter[feat] += 1

        for feat, count in selected_counter.items():
            stability_rows.append(
                {
                    'dataset': dataset_name,
                    'method': method_name,
                    'feature': feat,
                    'selected_count': count,
                    'total_runs': total_runs,
                    'stability_rate': count / total_runs,
                }
            )

selection_stability = pd.DataFrame(stability_rows)[required_columns_task2]
print(f'selection_stability shape: {selection_stability.shape}')
selection_stability.sort_values(['dataset', 'method', 'stability_rate'], ascending=[True, True, False]).head(10)

selection_stability shape: (48, 6)


,dataset,method,feature,selected_count,total_runs,stability_rate
36,finance,l1_logreg,num__loan_to_income,3,3,1.0
37,finance,l1_logreg,num__annual_income,3,3,1.0
38,finance,l1_logreg,num__credit_score,3,3,1.0
40,finance,l1_logreg,num__delinquency_count,3,3,1.0
42,finance,l1_logreg,cat__previous_default_yes,3,3,1.0
43,finance,l1_logreg,cat__previous_default_no,3,3,1.0
39,finance,l1_logreg,num__loan_amount,0,3,0.0
41,finance,l1_logreg,num__utilization_ratio,0,3,0.0
44,finance,l1_logreg,num__age,0,3,0.0
45,finance,l1_logreg,num__employment_years,0,3,0.0


### Задание 3. Формирование `set_D_robust` и экспорт

**Контракт задания**

Входные переменные:
- `method_agreement_long`, `selection_stability`, `feature_sets`.

Ожидаемый выход:
- обновленный `feature_sets[dataset]['set_D_robust']`;
- файлы `outputs/method_agreement_long.csv`, `outputs/selection_stability.csv`.

In [ ]:
# Что делаем: Загружаем входные данные и артефакты предыдущих шагов.
# Зачем: Этот шаг задает исходный контекст: без него метрики и графики будут считаться по неверным данным.
# Как читать результат: Проверьте размеры таблиц и названия ключевых колонок: это главный индикатор корректной загрузки.
# Типичные ошибки: Частая ошибка — использовать не тот файл или устаревший артефакт из другой лабораторной работы.
# 
method_agreement_path = OUTPUT_DIR / 'method_agreement_long.csv'
selection_stability_path = OUTPUT_DIR / 'selection_stability.csv'

required_columns_task1 = {'dataset', 'method_a', 'method_b', 'top_k', 'overlap_count', 'jaccard'}
required_columns_task2 = {'dataset', 'method', 'feature', 'selected_count', 'total_runs', 'stability_rate'}

# TODO(обязательно):
# 1) Убедитесь, что method_agreement_long и selection_stability содержат required columns.
# 2) Сформируйте set_D_robust (например, по порогу stability_rate >= 0.6).
# 3) Добавьте set_D_robust в feature_sets[dataset].
# 4) Сохраните два CSV-файла по указанным путям.


# Обязательный подпункт (методическая рефлексия):
# - До снятия intentional-stop добавьте в релевантный narrative-блок
#   или в `study-notes/*.md` короткое описание: что вы изучили,
#   с чем сравнили, на какие источники опирались.
# - Обновите `study-notes/glossary.md`: добавьте 2-3 термина, встретившихся в этом задании,
#   и укажите их простые объяснения + источники.

assert required_columns_task1.issubset(method_agreement_long.columns)
assert required_columns_task2.issubset(selection_stability.columns)
assert not method_agreement_long.empty
assert not selection_stability.empty

STABILITY_THRESHOLD = 0.6

for dataset_name in DATASETS:
    ds_stability = selection_stability[selection_stability['dataset'] == dataset_name]
    robust_feats = (
        ds_stability[ds_stability['stability_rate'] >= STABILITY_THRESHOLD]
        .sort_values('stability_rate', ascending=False)['feature']
        .tolist()
    )
    if len(robust_feats) < 5:
        robust_feats = (
            ds_stability.sort_values('stability_rate', ascending=False)['feature']
            .drop_duplicates()
            .head(10)
            .tolist()
        )
    feature_sets[dataset_name]['set_D_robust'] = robust_feats[:10]
    print(f"{dataset_name}: set_D_robust ({len(feature_sets[dataset_name]['set_D_robust'])} features)")

method_agreement_long.to_csv(method_agreement_path, index=False)
selection_stability.to_csv(selection_stability_path, index=False)

feature_sets_path = OUTPUT_DIR / 'feature_sets_wrapper_embedded.json'
with open(feature_sets_path, 'w', encoding='utf-8') as f:
    json.dump(feature_sets, f, ensure_ascii=False, indent=2)

print(f'Saved: {method_agreement_path}')
print(f'Saved: {selection_stability_path}')
print(f'Updated: {feature_sets_path}')

medical: set_D_robust (10 features)
finance: set_D_robust (10 features)
Saved: C:\Users\perev\Documents\GitHub\edu-big-data-machine-models\01-feature-importance-and-selection\outputs\method_agreement_long.csv
Saved: C:\Users\perev\Documents\GitHub\edu-big-data-machine-models\01-feature-importance-and-selection\outputs\selection_stability.csv
Updated: C:\Users\perev\Documents\GitHub\edu-big-data-machine-models\01-feature-importance-and-selection\outputs\feature_sets_wrapper_embedded.json
